# 📡 Stockbee Full US Market Screener

Scans ~6,700 US tickers for Stockbee/Pradeep Bonde momentum setups.

**Scans included:**
- Scan 1 — Extreme Sales Growth (99%+ YoY revenue)
- Scan 2 — Sales Growth (39%/39% 2Q avg) — *fail-closed: requires BOTH quarters*
- Scan 3 — Growth/Turnaround (≤$11B mktcap, IPO <10y, USD-quoted only)
- Scan 4 — Earnings + Sales (39%+ EPS both quarters + 20%+ revenue)
- Momentum — ATR Entry Zone (price ≥ MA70 + 2×ATR) + DI filter

**New vs prior version:**
- ✅ DAMA filter **removed** (Pine Script dropped it)
- ✅ 9M Vol flag now requires: vol ≥ 9M **AND** vol > prior day **AND** close up 4%+
- ✅ Plus4 now `flag_plus4_bo` (breakout): price +4% **AND** vol > prior day
- ✅ New `flag_minus4_bd` (breakdown): price -4% **AND** vol > prior day
- ✅ Scan 2 avgSalesChg2Q is **fail-closed** (both quarters required, no fallback)
- ✅ Scan 3 market cap check requires USD-quoted stock
- ✅ Gap detection fix: `not na(close[1])` guard added
- ✅ BMS Condition 1: candle body uses `open > 0` guard
- ✅ Momentum scan renamed from DAMA to ATR Entry Zone

⏱️ **Expected runtime:** 45–90 minutes for full universe  
💾 **Results auto-saved** to Google Drive every 50 tickers  
♻️ **Resumable:** re-run from checkpoint automatically if interrupted

**Run all cells: Runtime → Run all** (or Ctrl+F9)


In [ ]:
# @title 📦 Cell 1 — Install Dependencies
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                        'yfinance', 'pandas', 'numpy', 'requests', 'tqdm', '--quiet'])

import warnings, os, json, time, gc
from datetime import datetime
from typing import Optional
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import requests
import yfinance as yf
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

print('✅ Dependencies ready')
print(f'   pandas {pd.__version__}  |  yfinance {yf.__version__}')


In [ ]:
# @title ⚙️ Cell 2 — Configuration
# @markdown Adjust thresholds here to match Pine Script settings.

# ── Universe pre-filters ──
MIN_PRICE           = 10.0   # close >= $10 (all scans)
MIN_AVG_VOL_50K     = 100    # 50d avg volume >= 100K shares (scans 1/2/3)
MIN_AVG_VOL_200K    = 200    # 50d avg volume >= 200K shares (scan 4)

# ── EP Scan thresholds ──
EXTREME_SALES_PCT   = 99     # Scan 1: revenue YoY >= 99%
SALES_GROWTH_PCT    = 39     # Scan 2/3: revenue YoY >= 39% (last Q AND 2Q avg)
MIN_ANNUAL_SALES_M  = 25     # all scans: trailing 4Q revenue >= $25M
EPS_GROWTH_PCT      = 39     # Scan 4: EPS YoY >= 39% (BOTH quarters required)
SALES_GROWTH_EP4    = 20     # Scan 4: revenue YoY >= 20% (both quarters)
MKTCAP_SCAN3_B      = 11     # Scan 3: market cap <= $11B (USD only)
IPO_YEARS_SCAN3     = 10     # Scan 3: listed within 10 years

# ── Entry MA & ATR threshold ──
ENTRY_MA_PERIOD     = 70     # MA period for ATR entry threshold (Pine: entryMaPeriod)
ATR_MULTIPLE        = 2.0    # price must be >= MA + N*ATR to be in entry zone

# ── Momentum flags ──
TI65_BULL           = 1.05   # avgC7 / avgC65 >= 1.05
# 9M Vol: vol >= 9M AND vol > prior day AND close up 4%+ (Pine v2 definition)
VOL_9M              = 9_000_000
PLUS4PCT            = 4.0    # +4% breakout threshold

# ── Gap detection (shared by TTT & MWG) ──
# Gap bar: close > 1.2 * prior close AND (high-low) < 0.04 * close
GAP_JUMP_MULT       = 1.20
GAP_RANGE_PCT       = 0.04
GAP_LOOKBACK        = 100    # bars to look back for gap count

# ── Pattern filter toggles (set True to enable as hard filter) ──
USE_TTT_FILTER      = False
USE_MWG_FILTER      = False
USE_BMS_FILTER      = False
USE_DI_FILTER       = True   # require +DI > -DI (matches Pine default)

# ── Performance ──
MAX_WORKERS         = 8      # parallel threads
BATCH_SIZE          = 50     # checkpoint save every N tickers
HISTORY_PERIOD      = '18mo' # price history window

# ── Output ──
SAVE_TO_DRIVE       = True
RESULTS_FILE        = 'stockbee_results.csv'
HITS_FILE           = 'stockbee_hits.csv'

print('✅ Config loaded')


In [ ]:
# @title 🌐 Cell 3 — Build Full US Ticker Universe

def fetch_us_tickers() -> list:
    SKIP_SUFFIXES = ('W', 'R', 'U', 'Z', 'L')
    KNOWN_ETFS    = {'SPY','QQQ','IWM','DIA','GLD','SLV','TLT','HYG','LQD',
                     'XLB','XLC','XLE','XLF','XLI','XLK','XLP','XLU','XLV','XLY'}
    url = ('https://raw.githubusercontent.com/'
           'rreichel3/US-Stock-Symbols/main/all/all_tickers.txt')
    try:
        resp = requests.get(url, timeout=15)
        resp.raise_for_status()
        raw = [t.strip() for t in resp.text.split('\n') if t.strip()]
    except Exception as e:
        print(f'⚠️  Could not fetch ticker list ({e}). Using fallback.')
        return _fallback_tickers()

    clean = []
    for t in raw:
        if len(t) > 5:                          continue
        if '.' in t or '-' in t:               continue  # foreign/preferred
        if t in KNOWN_ETFS:                     continue
        if t.endswith(SKIP_SUFFIXES) and len(t) > 3: continue
        clean.append(t)
    print(f'✅ Universe: {len(clean):,} US tickers')
    return clean

def _fallback_tickers():
    return [
        'AAPL','MSFT','NVDA','TSLA','AMZN','META','GOOGL','AMD','AVGO','NFLX',
        'SMCI','PLTR','MSTR','IONQ','CRWD','SNOW','DDOG','ZS','MDB','BILL',
        'AFRM','UPST','SOFI','HOOD','RBLX','COIN','PATH','AI','GTLB','SOUN',
        'JOBY','ACHR','RKLB','ASTS','LUNR','RDDT','CELH','ELF','HIMS','IMVT',
        'AXSM','INSM','CAVA','BROS','WING','CMG','SHAK','SBUX','SHOP','MELI',
        'SE','GRAB','BABA','JD','PDD','TCOM','MNDY','CRM','NOW','VEEV',
        'HUBS','TEAM','ASAN','ZM','DOCU','TWLO','ESTC','OKTA','NET','CFLT',
        'RGEN','RXRX','PACB','NTLA','BEAM','CRSP','EDIT','EXAS','TMDX','INSP',
        'ACMR','ONTO','FORM','AZTA','AMBA','MCHP','MPWR','ALGM','ENPH','FSLR',
        'RIVN','LCID','NIO','LI','GM','F','JPM','BAC','WFC','GS',
        'REGN','BIIB','VRTX','ALNY','MRNA','BNTX','GILD','AMGN','ABBV','LLY',
        'MARA','RIOT','CLSK','WULF','IREN','COST','WMT','TGT','EBAY','ETSY',
    ]

ALL_TICKERS = fetch_us_tickers()
print(f'   Ready to scan {len(ALL_TICKERS):,} tickers')


In [ ]:
# @title 🔧 Cell 4 — Calculation Engine (aligned to Pine Script v2)

# ── Helpers ──────────────────────────────────────────────────────────

def safe_pct(new_val, old_val) -> Optional[float]:
    """YoY % change. Uses abs(old_val) as denominator (matches Pine for EPS)."""
    if pd.isna(new_val) or pd.isna(old_val) or old_val == 0:
        return None
    return ((new_val - old_val) / abs(old_val)) * 100

def safe_pct_pos_base(new_val, old_val) -> Optional[float]:
    """Revenue YoY — Pine requires old_val > 0 (not just != 0)."""
    if pd.isna(new_val) or pd.isna(old_val) or old_val <= 0:
        return None
    return ((new_val - old_val) / old_val) * 100

def calc_atr(high, low, close, period=14):
    tr = pd.concat([
        high - low,
        (high - close.shift(1)).abs(),
        (low  - close.shift(1)).abs(),
    ], axis=1).max(axis=1)
    return tr.ewm(alpha=1/period, adjust=False).mean()

def count_gaps(close, high, low, lookback=100):
    """
    Gap bar: close > 1.2 * prior close AND (high-low) < 0.04 * close.
    Pine fix: requires prior close is not NaN (not na(close[1]) guard).
    Returns rolling sum of gap bars over lookback window.
    """
    prior_close = close.shift(1)
    is_gap = (
        prior_close.notna() &
        (close > GAP_JUMP_MULT * prior_close) &
        ((high - low) < GAP_RANGE_PCT * close)
    ).astype(int)
    return is_gap.rolling(lookback, min_periods=lookback).sum()

def get_fundamentals(tk) -> dict:
    """Pull quarterly revenue & EPS from yfinance."""
    out = dict(rev_q=[], eps_q=[], annual_sales_m=None,
               market_cap=None, currency=None, ipo_years_ago=None)
    try:
        fi = tk.fast_info
        out['market_cap'] = getattr(fi, 'market_cap', None)
        out['currency']   = getattr(fi, 'currency', None)
    except Exception:
        pass

    try:
        fins = tk.quarterly_financials
        if fins is not None and not fins.empty:
            for label in ['Total Revenue', 'TotalRevenue', 'Revenue']:
                if label in fins.index:
                    row = fins.loc[label].sort_index(ascending=False)
                    out['rev_q'] = [v for v in row.values if not pd.isna(v)]
                    break
    except Exception:
        pass

    try:
        eps_df = tk.quarterly_earnings
        if eps_df is not None and not eps_df.empty and 'Reported EPS' in eps_df.columns:
            eps_s = eps_df.sort_index(ascending=False)['Reported EPS']
            out['eps_q'] = [v for v in eps_s.values if not pd.isna(v)]
    except Exception:
        pass

    if len(out['rev_q']) >= 4:
        out['annual_sales_m'] = sum(out['rev_q'][:4]) / 1_000_000

    try:
        info = tk.info
        raw  = info.get('firstTradeDateEpochUtc') or info.get('ipoExpectedDate')
        if raw:
            dt = datetime.fromtimestamp(raw) if isinstance(raw, (int, float)) else pd.to_datetime(raw)
            out['ipo_years_ago'] = (datetime.now() - dt).days / 365.25
    except Exception:
        pass

    return out


# ── Pattern filters (Ants TTT, MWG, BMS) ────────────────────────────

def calc_ttt(close, high, low, volume,
             min_vol=300_000, min_price=10.0,
             max_pct_3bar=1.5, max_pct_today=0.3, lookback=100):
    """
    Ants TTT (Tight-Tight-Tight):
    - min volume in last 3 days >= threshold
    - |3-bar price change| <= 1.5%
    - |today change| <= 0.3%
    - no gap bars in lookback window (fail-closed until lookback bars exist)
    """
    min_vol_3d   = volume.shift(1).rolling(3).min()
    vol_ok       = min_vol_3d >= min_vol
    price_ok     = close > min_price
    pct_3bar     = ((close - close.shift(3)) / close.shift(3) * 100).abs()
    consol_ok    = pct_3bar <= max_pct_3bar
    pct_today    = ((close - close.shift(1)) / close.shift(1) * 100).abs()
    tight_ok     = pct_today <= max_pct_today
    gap_sum      = count_gaps(close, high, low, lookback)
    no_gaps_ok   = gap_sum == 0  # NaN until lookback bars → False (fail-closed)
    return (vol_ok & price_ok & consol_ok & tight_ok & no_gaps_ok).fillna(False)

def calc_mwg(close, high, low, volume,
             min_vol=100_000, vol_days=3, min_price=10.0,
             max_day_pct=0.4, lookback=100):
    """
    Ants Bullish / MWG (Momentum Without Gaps):
    - momentum: close >= 1.2*30d low OR avg7/avg65 >= 1.05
    - controlled move: |today %| <= 0.4%
    - min volume in N days >= threshold
    - no gaps in lookback (fail-closed)
    """
    low30      = close.rolling(30).min()
    avg7       = close.rolling(7).mean()
    avg65      = close.rolling(65).mean()
    mom_ok     = (close / low30 >= 1.20) | (avg7 / avg65 >= 1.05)
    price_ok   = close > min_price
    pct_today  = ((close - close.shift(1)) / close.shift(1) * 100).abs()
    ctrl_ok    = pct_today <= max_day_pct
    min_vol_nd = volume.shift(1).rolling(vol_days).min()
    vol_ok     = min_vol_nd >= min_vol
    gap_sum    = count_gaps(close, high, low, lookback)
    no_gaps_ok = gap_sum == 0
    return (mom_ok & price_ok & ctrl_ok & vol_ok & no_gaps_ok).fillna(False)

def calc_bms(close, high, low, volume, open_,
             min_price=10.0, min_vol_c1=1_000_000, min_vol_c2=100_000,
             breakout_pct=4.0, stability_pct=2.0, close_str_min=0.70):
    """
    Bullish Combo (BMS):
    Condition 1: 2%+ candle body (with open>0 guard), vol>1M, today range >= prior, prior day stable
    Condition 2: price up 4%+, vol surge, close strength >= 70%
    """
    # Pine fix: open > 0 guard on candle body calculation
    candle_body  = ((close - open_) / open_.clip(lower=0.01)) >= 0.02
    today_range  = high - low
    prior_range  = (high.shift(1) - low.shift(1)).abs()
    continuation = today_range >= prior_range
    # Pine: stability uses close[1]/close[2] ratio
    prior_chg    = ((close.shift(1) / close.shift(2)) - 1).abs()
    stability    = prior_chg <= (stability_pct / 100)
    cond1 = candle_body & (volume > min_vol_c1) & continuation & stability

    breakout     = (close / close.shift(1)) >= (1 + breakout_pct / 100)
    vol_surge    = volume > volume.shift(1)
    cond2 = breakout & vol_surge & (volume >= min_vol_c2) & stability

    price_ok  = close >= min_price
    rng       = (high - low).clip(lower=1e-9)
    close_str = (close - low) / rng
    cs_ok     = close_str >= close_str_min

    return ((cond1 | cond2) & price_ok & cs_ok).fillna(False)


# ── Main single-ticker screener ───────────────────────────────────────

def screen_one(sym: str) -> dict:
    row = dict(
        ticker=sym, price=None,
        # EP scans
        scan1=False, scan2=False, scan3=False, scan4=False,
        # Momentum / entry zone
        momentum=False,
        # Pattern filters (stored even if not used as hard filter)
        ttt_pass=False, mwg_pass=False, bms_pass=False,
        # DI
        di_bullish=False,
        # Flags
        flag_ti65=False, flag_9mvol=False,
        flag_plus4_bo=False,   # NEW: +4% breakout (price up + vol > prior)
        flag_minus4_bd=False,  # NEW: -4% breakdown (price down + vol > prior)
        # Metrics
        ti65=None, dama_126=None,
        sales_pct=None, eps_pct=None,
        annual_sales_m=None, mktcap_b=None,
        avg_vol_50k=None, adr_pct=None, ext_atr=None,
        error=None,
    )
    try:
        tk   = yf.Ticker(sym)
        hist = tk.history(period=HISTORY_PERIOD, interval='1d', auto_adjust=True)
        if hist.empty or len(hist) < 70:
            row['error'] = 'insufficient_history'; return row

        close  = hist['Close']
        high   = hist['High']
        low    = hist['Low']
        volume = hist['Volume']
        open_  = hist['Open']

        last_close = float(close.iloc[-1])
        last_vol   = float(volume.iloc[-1])
        avg50      = float(volume.rolling(50).mean().iloc[-1])
        avg50k     = avg50 / 1000

        # Pre-filter: price and min volume (skip cheap/illiquid)
        if last_close < MIN_PRICE or avg50k < MIN_AVG_VOL_50K:
            row['error'] = 'filtered'; return row

        row['price']       = round(last_close, 2)
        row['avg_vol_50k'] = round(avg50k, 1)

        # ── ATR(14) ──
        atr14  = calc_atr(high, low, close, 14)
        latr   = float(atr14.iloc[-1])

        # ── Entry MA (MA70 default) ──
        entry_ma  = close.rolling(ENTRY_MA_PERIOD).mean()
        lma       = float(entry_ma.iloc[-1])

        # ── ATR Entry Zone: price >= MA + ATR_MULTIPLE * ATR ──
        # NOTE: DAMA filter removed from Pine Script — only entry zone used now
        in_entry_zone = last_close >= lma + ATR_MULTIPLE * latr

        # ── ADR% ──
        row['adr_pct'] = round(
            float(((high - low) / close * 100).rolling(14).mean().iloc[-1]), 2
        )

        # ── Extension from MA70 in ATR units ──
        ma70  = close.rolling(70).mean()
        row['ext_atr'] = round(
            (last_close - float(ma70.iloc[-1])) / latr, 2
        ) if latr > 0 else None

        # ── TI65: avgC7 / avgC65 ──
        avg7   = close.rolling(7).mean()
        avg65  = close.rolling(65).mean()
        ti65_s = avg7 / avg65
        lti65  = float(ti65_s.iloc[-1]) if not pd.isna(ti65_s.iloc[-1]) else 0
        row['ti65']      = round(lti65, 3)
        row['flag_ti65'] = lti65 >= TI65_BULL

        # ── DI filter: +DI > -DI using EWM approximation ──
        # True Wilder DMI needs more logic; use slope proxy
        # Close above MA70 is a simple directional confirmation proxy
        row['di_bullish'] = last_close > lma

        # ── 9M Vol flag (UPDATED Pine v2 definition) ──
        # Pine: volume >= 9M AND volume > volume[1] AND close >= 1.04 * close[1]
        prior_close = float(close.iloc[-2]) if len(close) >= 2 else last_close
        prior_vol   = float(volume.iloc[-2]) if len(volume) >= 2 else last_vol
        row['flag_9mvol'] = (
            last_vol >= VOL_9M
            and last_vol > prior_vol
            and last_close >= 1.04 * prior_close
        )

        # ── +4% Breakout flag (UPDATED Pine v2 definition) ──
        # Pine: pctChangeToday >= 4.0 AND volume > volume[1]
        pct_today = ((last_close - prior_close) / prior_close * 100) if prior_close else 0
        row['flag_plus4_bo']  = pct_today >= PLUS4PCT and last_vol > prior_vol

        # ── -4% Breakdown flag (NEW in Pine v2) ──
        # Pine: pctChangeToday <= -4.0 AND volume > volume[1]
        row['flag_minus4_bd'] = pct_today <= -PLUS4PCT and last_vol > prior_vol

        # ── Pattern filters (computed, used as hard filter only if USE_*=True) ──
        row['ttt_pass'] = bool(calc_ttt(close, high, low, volume).iloc[-1])
        row['mwg_pass'] = bool(calc_mwg(close, high, low, volume).iloc[-1])
        row['bms_pass'] = bool(calc_bms(close, high, low, volume, open_).iloc[-1])

        passes_ttt = row['ttt_pass'] if USE_TTT_FILTER else True
        passes_mwg = row['mwg_pass'] if USE_MWG_FILTER else True
        passes_bms = row['bms_pass'] if USE_BMS_FILTER else True
        passes_di  = row['di_bullish'] if USE_DI_FILTER else True

        # ── Momentum / ATR Entry Zone ──
        # Pine v2: DAMA removed; entry zone = price >= MA + 2*ATR + DI + pattern filters
        row['momentum'] = (
            in_entry_zone
            and passes_di
            and passes_ttt
            and passes_mwg
            and passes_bms
        )

        # ── Fundamentals ──
        fin  = get_fundamentals(tk)
        revq = fin['rev_q']
        epsq = fin['eps_q']
        ann  = fin['annual_sales_m']
        mcap = fin['market_cap']
        curr = fin['currency']
        ipo  = fin['ipo_years_ago']

        row['annual_sales_m'] = round(ann, 1)      if ann  else None
        row['mktcap_b']       = round(mcap/1e9, 2) if mcap else None

        # Revenue YoY — Pine requires old_val > 0 (positive base only)
        sg   = safe_pct_pos_base(revq[0], revq[4]) if len(revq) >= 5 else None
        sg1  = safe_pct_pos_base(revq[1], revq[5]) if len(revq) >= 6 else None
        # UPDATED: avgSalesChg2Q is FAIL-CLOSED — requires BOTH quarters
        # Pine v2: no single-quarter fallback
        sg2  = (sg + sg1) / 2 if (sg is not None and sg1 is not None) else None
        row['sales_pct'] = round(sg, 1) if sg is not None else None

        # EPS YoY — Pine uses abs(prior) as denominator
        eg   = safe_pct(epsq[0], epsq[4]) if len(epsq) >= 5 else None
        eg1  = safe_pct(epsq[1], epsq[5]) if len(epsq) >= 6 else None
        row['eps_pct'] = round(eg, 1) if eg is not None else None

        # ── Common pre-checks ──
        price_ok    = last_close >= MIN_PRICE
        vol100_ok   = avg50k >= MIN_AVG_VOL_50K
        vol200_ok   = avg50k >= MIN_AVG_VOL_200K
        sales_ok    = ann is not None and ann >= MIN_ANNUAL_SALES_M

        # ── Scan 1: Extreme Sales (99%+ YoY) ──
        row['scan1'] = (
            sg is not None and sg >= EXTREME_SALES_PCT
            and sales_ok and price_ok and vol100_ok
        )

        # ── Scan 2: Sales Growth 39/39 ──
        # UPDATED: salesCoreOK shared base (Pine v2 pattern)
        # avgSalesChg2Q MUST be non-None (fail-closed — both quarters required)
        sales_core_ok = (
            sg  is not None and sg  >= SALES_GROWTH_PCT
            and sg2 is not None and sg2 >= SALES_GROWTH_PCT  # fail-closed
            and sales_ok and price_ok and vol100_ok
        )
        row['scan2'] = sales_core_ok

        # ── Scan 3: Growth/Turnaround ──
        # UPDATED: market cap check requires USD currency (Pine: syminfo.currency == 'USD')
        usd_ok    = curr == 'USD' or curr is None  # be slightly permissive if unknown
        small_mid = mcap is not None and mcap <= MKTCAP_SCAN3_B * 1e9 and usd_ok
        new_ipo   = ipo is None or ipo <= IPO_YEARS_SCAN3
        row['scan3'] = sales_core_ok and small_mid and new_ipo

        # ── Scan 4: EPS + Sales ──
        # UPDATED: BOTH EPS quarters required (eg AND eg1 must be non-None)
        # UPDATED: avgSalesChg2Q also fail-closed here
        sg2_ep4 = (sg + sg1) / 2 if (sg is not None and sg1 is not None) else None
        row['scan4'] = (
            eg  is not None and eg  >= EPS_GROWTH_PCT
            and eg1 is not None and eg1 >= EPS_GROWTH_PCT  # both quarters
            and sg  is not None and sg  >= SALES_GROWTH_EP4
            and sg2_ep4 is not None and sg2_ep4 >= SALES_GROWTH_EP4  # fail-closed
            and sales_ok and price_ok and vol200_ok
        )

    except Exception as exc:
        row['error'] = str(exc)[:60]

    return row

print('✅ Calculation engine ready')


In [ ]:
# @title 💾 Cell 5 — Drive Mount + Scan Runner

def mount_drive_if_colab():
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        save_dir = '/content/drive/MyDrive/Stockbee'
        os.makedirs(save_dir, exist_ok=True)
        print(f'✅ Google Drive mounted → {save_dir}/')
        return save_dir
    except ImportError:
        print('ℹ️  Not in Colab — saving locally')
        return '.'
    except Exception as e:
        print(f'⚠️  Drive mount failed ({e}) — saving locally')
        return '.'


def _save_checkpoint(results, cp_path, hits_path, results_path):
    try:
        df_cp = pd.DataFrame(results)
        scan_cols = [c for c in ['scan1','scan2','scan3','scan4','momentum'] if c in df_cp]
        df_cp['any_hit'] = df_cp[scan_cols].any(axis=1)
        df_cp.to_csv(cp_path, index=False)
        df_cp[df_cp['any_hit']].to_csv(hits_path, index=False)
    except Exception:
        pass


def run_full_scan(tickers: list, save_dir: str = '.') -> pd.DataFrame:
    results_path = os.path.join(save_dir, RESULTS_FILE)
    hits_path    = os.path.join(save_dir, HITS_FILE)
    cp_path      = os.path.join(save_dir, 'stockbee_checkpoint.csv')

    results, done_tickers = [], set()
    if os.path.exists(cp_path):
        try:
            prev = pd.read_csv(cp_path)
            results = prev.to_dict('records')
            done_tickers = set(prev['ticker'].tolist())
            print(f'♻️  Resuming: {len(done_tickers)} already done')
        except Exception:
            pass

    remaining  = [t for t in tickers if t not in done_tickers]
    start_time = time.time()
    print(f'\n{"━"*60}')
    print(f'  STOCKBEE FULL US MARKET SCAN')
    print(f'  {datetime.now().strftime("%Y-%m-%d %H:%M")}  |  {len(remaining):,} tickers')
    print(f'  Threads: {MAX_WORKERS}  |  Checkpoint every {BATCH_SIZE}')
    print(f'{"━"*60}\n')

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(screen_one, sym): sym for sym in remaining}
        with tqdm(total=len(remaining), unit='ticker',
                  bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]') as pbar:
            for i, future in enumerate(as_completed(futures), 1):
                sym = futures[future]
                try:
                    row = future.result(timeout=30)
                except Exception as e:
                    row = {k: False for k in ['scan1','scan2','scan3','scan4','momentum',
                           'flag_ti65','flag_9mvol','flag_plus4_bo','flag_minus4_bd']}
                    row.update(ticker=sym, error=str(e)[:40])
                results.append(row)
                pbar.set_postfix({'last': sym}, refresh=False)
                pbar.update(1)
                if i % BATCH_SIZE == 0:
                    _save_checkpoint(results, cp_path, hits_path, results_path)
                    elapsed = time.time() - start_time
                    eta_s   = (len(remaining) - i) / (i / elapsed) if elapsed else 0
                    pbar.set_postfix({'ETA': f'{int(eta_s//60)}m{int(eta_s%60)}s'})
                gc.collect()

    df = pd.DataFrame(results)
    df['any_hit'] = df[['scan1','scan2','scan3','scan4','momentum']].any(axis=1)
    df.to_csv(results_path, index=False)
    df[df['any_hit']].to_csv(hits_path, index=False)

    elapsed = time.time() - start_time
    hits = df['any_hit'].sum()
    print(f'\n{"━"*60}')
    print(f'  ✅ COMPLETE  |  {elapsed/60:.1f} min')
    print(f'  Scanned: {len(df):,}  |  Hits: {hits}')
    print(f'  Saved → {results_path}')
    print(f'{"━"*60}\n')
    return df

print('✅ Scanner ready')


In [ ]:
# @title 📊 Cell 6 — Display Results

def display_results(df: pd.DataFrame):
    scan_defs = [
        ('scan1',    'SCAN 1',    'Extreme Sales Growth (99%+ YoY)'),
        ('scan2',    'SCAN 2',    'Sales Growth (39%/39% — fail-closed 2Q)'),
        ('scan3',    'SCAN 3',    'Growth/Turnaround (<=11B, USD, IPO <10y)'),
        ('scan4',    'SCAN 4',    'Earnings + Sales (39%E both Q / 20%S)'),
        ('momentum', 'MOMENTUM',  'ATR Entry Zone (MA70 + 2xATR + DI)'),
    ]
    cols = ['ticker','price','sales_pct','eps_pct','annual_sales_m',
            'mktcap_b','avg_vol_50k','ti65','adr_pct','ext_atr']

    total_hits = int(df['any_hit'].sum()) if 'any_hit' in df.columns else 0
    print(f'\n{"="*80}')
    print(f'  STOCKBEE RESULTS  |  {datetime.now().strftime("%Y-%m-%d %H:%M")}')
    print(f'  Universe: {len(df):,}  |  Hits: {total_hits}')
    print(f'{"="*80}')

    for key, tag, label in scan_defs:
        if key not in df.columns: continue
        subset = df[df[key]].copy()
        if subset.empty:
            print(f'\n  [{tag}] {label}  →  No hits'); continue
        subset = subset.sort_values('sales_pct', ascending=False, na_position='last')
        subset['flags'] = (
            subset['flag_9mvol'].apply(lambda x: '🔊' if x else '') +
            subset['flag_plus4_bo'].apply(lambda x: '🚀' if x else '') +
            subset['flag_minus4_bd'].apply(lambda x: '🔻' if x else '') +
            subset['flag_ti65'].apply(lambda x: '📈' if x else '')
        )
        print(f'\n  ▌ [{tag}] {label}  ({len(subset)} stocks)')
        display_cols = [c for c in cols if c in subset.columns] + ['flags']
        try:
            from IPython.display import display as ipy_display
            ipy_display(subset[display_cols].reset_index(drop=True))
        except ImportError:
            print(subset[display_cols].to_string(index=False))

    print(f'\n  Flags: 🔊 9M+vol+4%up  🚀 +4% BO  🔻 -4% BD  📈 TI65 bullish\n')


def top_momentum_stocks(df: pd.DataFrame, n: int = 20) -> pd.DataFrame:
    """Top N stocks by TI65 score in entry zone."""
    if 'momentum' not in df.columns: return df.head(0)
    mom = df[df['momentum']].copy()
    mom['score'] = mom['ti65'].fillna(0)
    return mom.sort_values('score', ascending=False).head(n)

print('✅ Display functions ready')


In [ ]:
# @title ▶️ Cell 7 — RUN EVERYTHING
# @markdown ⏱️ ~45-90 min for full universe. Saves to Google Drive automatically.

# Mount Drive
save_dir = mount_drive_if_colab() if SAVE_TO_DRIVE else '.'

# Run scan
df = run_full_scan(ALL_TICKERS, save_dir=save_dir)

# Display detailed results
df['any_hit'] = df[['scan1','scan2','scan3','scan4','momentum']].any(axis=1)
display_results(df)

# Top momentum
print('\n  🏆 TOP 20 MOMENTUM STOCKS (ATR Entry Zone, ranked by TI65)\n')
top = top_momentum_stocks(df, 20)
try:
    from IPython.display import display as ipy_display
    ipy_display(top[['ticker','price','ti65','ext_atr','adr_pct']].reset_index(drop=True))
except Exception:
    print(top[['ticker','price','ti65','ext_atr']].to_string(index=False))

print(f'\n  📁 Saved to: {save_dir}/')
print(f'     • {HITS_FILE}  — hits only')
print(f'     • {RESULTS_FILE}  — full universe\n')


---
## 📋 Final Summary — Ticker Lists by Group
Run the cell below after the scan completes. Produces comma-separated ticker lists per scan group — ready to paste into TradingView, your broker watchlist, or a spreadsheet.

In [ ]:
# @title 📋 Cell 8 — Final Summary: Comma-Separated Ticker Lists

from IPython.display import display, HTML

def print_summary(df):
    scan_defs = [
        ('scan1',    'SCAN 1',    'Extreme Sales Growth',   '99%+ YoY revenue',                '#f59e0b'),
        ('scan2',    'SCAN 2',    'Sales Growth',           '39%/39% fail-closed 2Q avg',       '#10b981'),
        ('scan3',    'SCAN 3',    'Growth / Turnaround',    '<=11B USD mktcap + IPO <10y',      '#a78bfa'),
        ('scan4',    'SCAN 4',    'Earnings + Sales',       '39%E both Q + 20%S fail-closed',   '#00e5ff'),
        ('momentum', 'MOMENTUM', 'ATR Entry Zone',         'MA70 + 2xATR + DI + patterns',     '#ef4444'),
    ]

    scan_cols = [s[0] for s in scan_defs if s[0] in df.columns]
    all_hits  = sorted(df[df[scan_cols].any(axis=1)]['ticker'].dropna().unique().tolist())

    print('\n' + '=' * 70)
    print(f'  STOCKBEE SCREENER SUMMARY  |  {datetime.now().strftime("%Y-%m-%d %H:%M")}')
    print(f'  Universe : {len(df):,} tickers scanned')
    print(f'  Unique hits : {len(all_hits)} stocks')
    print('=' * 70)

    group_tickers = {}
    for key, tag, name, criteria, _ in scan_defs:
        if key not in df.columns: continue
        tickers = (
            df[df[key]]
            .sort_values('sales_pct', ascending=False, na_position='last')['ticker']
            .dropna().unique().tolist()
        )
        group_tickers[key] = tickers
        print(f'\n  [{tag}] {name} — {criteria}')
        print(f'  Count   : {len(tickers)}')
        print(f'  Tickers : {", ".join(tickers) if tickers else "(none)"}')

    # Flag sub-lists (within hits only)
    flag_defs = [
        ('flag_9mvol',     '🔊', '9M Vol + price up 4%+ (EP signal)'),
        ('flag_plus4_bo',  '🚀', '+4% Breakout (vol confirmed)'),
        ('flag_minus4_bd', '🔻', '-4% Breakdown (vol confirmed)'),
        ('flag_ti65',      '📈', 'TI65 Bullish (avg7 >= 1.05x avg65)'),
    ]
    print()
    for flag, emoji, label in flag_defs:
        if flag not in df.columns: continue
        fl = sorted(df[df[flag] & df[scan_cols].any(axis=1)]['ticker'].dropna().unique().tolist())
        print(f'  {emoji} {label} ({len(fl)}) : {", ".join(fl) if fl else "(none)"}')

    print(f'\n  ALL HITS (unique, A-Z) [{len(all_hits)}]:')
    print(f'  {", ".join(all_hits)}')
    print('=' * 70)

    # ── HTML card ──
    rows_html = ''
    for key, tag, name, criteria, color in scan_defs:
        tickers = group_tickers.get(key, [])
        chips = ' '.join(
            f'<span style="background:{color}22;border:1px solid {color}66;color:{color};'
            f'padding:2px 8px;border-radius:4px;font-family:monospace;font-size:13px;'
            f'margin:2px;display:inline-block">{t}</span>'
            for t in tickers
        ) or '<span style="color:#555">No hits</span>'
        rows_html += f'''
        <tr>
          <td style="padding:10px 14px;border-bottom:1px solid #1e293b;white-space:nowrap;vertical-align:top">
            <span style="background:{color}22;border:1px solid {color}55;color:{color};
              padding:3px 9px;border-radius:4px;font-family:monospace;font-weight:bold;font-size:12px">{tag}</span>
            <div style="color:#94a3b8;font-size:12px;margin-top:5px">{name}</div>
            <div style="color:#475569;font-size:11px">{criteria}</div>
          </td>
          <td style="padding:10px 14px;border-bottom:1px solid #1e293b;text-align:center;
            font-family:monospace;font-size:16px;color:#e2e8f0;vertical-align:top">{len(tickers)}</td>
          <td style="padding:10px 14px;border-bottom:1px solid #1e293b;vertical-align:top">{chips}</td>
        </tr>'''

    all_chips = ' '.join(
        f'<span style="background:#1e293b;border:1px solid #334155;color:#cbd5e1;'
        f'padding:2px 8px;border-radius:4px;font-family:monospace;font-size:13px;'
        f'margin:2px;display:inline-block">{t}</span>'
        for t in all_hits
    ) or '<span style="color:#555">No hits</span>'

    display(HTML(f'''
    <div style="background:#0f172a;border-radius:12px;padding:24px;margin-top:16px;font-family:sans-serif">
      <div style="color:#00e5ff;font-family:monospace;font-size:20px;font-weight:700">
        📡 STOCKBEE — FINAL SUMMARY
      </div>
      <div style="color:#475569;font-size:12px;margin:6px 0 20px">
        {datetime.now().strftime("%Y-%m-%d %H:%M")} &nbsp;·&nbsp;
        {len(df):,} tickers scanned &nbsp;·&nbsp;
        <b style="color:#94a3b8">{len(all_hits)} unique hits</b>
      </div>
      <table style="width:100%;border-collapse:collapse;border-radius:8px;overflow:hidden;border:1px solid #1e293b">
        <thead>
          <tr style="background:#1e293b">
            <th style="padding:10px 14px;text-align:left;color:#475569;font-size:11px;
              letter-spacing:.1em;text-transform:uppercase;border-bottom:1px solid #0f172a">Scan</th>
            <th style="padding:10px 14px;text-align:center;color:#475569;font-size:11px;
              letter-spacing:.1em;text-transform:uppercase;border-bottom:1px solid #0f172a">#</th>
            <th style="padding:10px 14px;text-align:left;color:#475569;font-size:11px;
              letter-spacing:.1em;text-transform:uppercase;border-bottom:1px solid #0f172a">Tickers (by Sales Growth)</th>
          </tr>
        </thead>
        <tbody style="background:#0f172a">
          {rows_html}
          <tr style="background:#1e293b">
            <td style="padding:12px 14px;vertical-align:top">
              <span style="color:#e2e8f0;font-family:monospace;font-weight:700">ALL HITS</span>
              <div style="color:#475569;font-size:11px;margin-top:3px">unique · A-Z</div>
            </td>
            <td style="padding:12px 14px;text-align:center;font-family:monospace;
              font-size:18px;font-weight:700;color:#00e5ff;vertical-align:top">{len(all_hits)}</td>
            <td style="padding:12px 14px;vertical-align:top">{all_chips}</td>
          </tr>
        </tbody>
      </table>
    </div>
    '''))

print_summary(df)


In [ ]:
# ── SAVE TO GOOGLE DRIVE folder: scan_result ──────────────────────────
# Folder: https://drive.google.com/drive/folders/1qFAeut_82HvPP2-tntvxmxVl-uysms9c
# This cell runs independently — no drive.mount needed
import os, pandas as pd
from datetime import datetime
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaInMemoryUpload

auth.authenticate_user()
from google.auth import default
_creds, _ = default()
_svc = build('drive', 'v3', credentials=_creds)
_FOLDER_ID = "1qFAeut_82HvPP2-tntvxmxVl-uysms9c"

def upload_csv(df, filename):
    csv_bytes = df.to_csv(index=False).encode('utf-8')
    media = MediaInMemoryUpload(csv_bytes, mimetype='text/csv', resumable=False)
    q = "name='" + filename + "' and '" + _FOLDER_ID + "' in parents and trashed=false"
    existing = _svc.files().list(q=q, fields='files(id,name)').execute().get('files', [])
    if existing:
        _svc.files().update(fileId=existing[0]['id'], media_body=media).execute()
        print("  Updated:", filename, "->", len(df), "rows ->", "https://drive.google.com/drive/folders/1qFAeut_82HvPP2-tntvxmxVl-uysms9c")
    else:
        meta = dict(name=filename, parents=[_FOLDER_ID])
        _svc.files().create(body=meta, media_body=media, fields='id').execute()
        print("  Created:", filename, "->", len(df), "rows ->", "https://drive.google.com/drive/folders/1qFAeut_82HvPP2-tntvxmxVl-uysms9c")

# df = full results DataFrame from run_stockbee_scan()
_hits = df[df['any_hit'] == True].copy() if 'any_hit' in df.columns else df.copy()
_hits['Updated'] = datetime.now().strftime('%Y-%m-%d %H:%M')
upload_csv(_hits, 'stockbee_hits.csv')
